<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 6 — Fine-tuning de BERT</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">Un backbone, tres cabezas — con datasets reales y demo sobre su corpus</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.


> ⚙️ **Entorno: Google Colab con GPU T4 (free tier).** Entorno de ejecución → Cambiar tipo → **GPU**. 
Los tres modelos (BETO 110M, Sentence-BERT) caben de sobra en los 15 GB de la T4 usando `fp16`. 
Esta versión entrena con **datasets reales de HuggingFace** (no con el corpus de juguete): `tweet_sentiment_multilingual` para clasificación y `conll2002` para NER. El corpus chiapaneco se usa solo para la **inferencia final** de cada parte. Las celdas de *liberar memoria* permiten correr A → B → C en una sola sesión sin saturar la VRAM.


## Objetivo

Tres partes, el mismo BERT preentrenado en español como base. **A)** Afinar un encoder de oraciones (Sentence-BERT) con sus pares del Lab 3. **B)** Clasificar **sentimiento** con un dataset real en español. **C)** **NER** con CoNLL-2002. Cada parte cierra **usando el modelo afinado** sobre el corpus chiapaneco, y libera memoria antes de la siguiente.


## 0 · Setup, GPU y utilidades

In [ ]:
# Instalación (Colab). Reiniciar el entorno si lo pide tras instalar.
!pip install -q transformers sentence-transformers datasets seqeval accelerate

import gc, math, json
import numpy as np
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — activen el runtime de GPU')

def liberar_memoria():
    """Libera RAM/VRAM. Llamar tras borrar (del) las variables del entrenamiento previo."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f'VRAM en uso: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# El corpus chiapaneco (del Lab 1) se usa SOLO para la inferencia final de cada parte.
with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)
with open('corpus_crudo.json', encoding='utf-8') as fh:
    crudo = {d['id']: d['texto'] for d in json.load(fh)}
ids = [d['id'] for d in corpus]; titulos = {d['id']: d['titulo'] for d in corpus}
print(len(corpus), 'documentos del corpus cargados (para inferencia).')

---
## Parte A · Embeddings con Sentence-BERT (datos: sus qrels del Lab 3)

> **Por qué aquí sí usamos el corpus.** El fine-tuning de embeddings necesita pares *de dominio* (consulta ↔ documento relevante). Esos pares salen de **sus** qrels del Lab 3: es el cierre del arco de la unidad, donde su juicio de relevancia se vuelve señal de entrenamiento. **Advertencia metodológica:** con ~5 consultas esto es una *demostración del método*, no un experimento — la mejora puede ser chica o ruidosa. Lo evaluable es el pipeline correcto, no el número.


**A.1** Línea base: carguen un Sentence-BERT en español y midan su buscador **sin afinar** con su nDCG del Lab 3.

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

# Cargar el modelo base preentrenado en español para similitud
modelo = SentenceTransformer('hiiamsid/sentence_similarity_spanish_es')

# 1. Recuperar las estructuras de referencia del Lab 3
qrels = {
    'sequia y cultivos de maiz': {'d04': 3, 'd02': 2},
    'problemas de agua': {'d13': 3, 'd02': 2, 'd01': 1},
    'turismo viajeros destinos': {'d05': 3, 'd09': 3}
}

def emb_corpus():
    # Generar embeddings de todos los títulos del corpus usando el modelo actual
    return {d_id: modelo.encode(titulos[d_id], convert_to_numpy=True) for d_id in ids}

def buscar(consulta, EMB, k=5):
    q_vec = modelo.encode(consulta, convert_to_numpy=True)
    scores = []
    for d_id, d_vec in EMB.items():
        dot = np.dot(q_vec, d_vec)
        norm_q = np.linalg.norm(q_vec)
        norm_d = np.linalg.norm(d_vec)
        coseno = dot / (norm_q * norm_d) if (norm_q > 0 and norm_d > 0) else 0.0
        scores.append((d_id, titulos[d_id], coseno))
    return sorted(scores, key=lambda x: -x[2])[:k]

def ndcg_medio(EMB):
    suma_ndcg = 0.0
    for qid, rels in qrels.items():
        ranking = buscar(qid, EMB, k=5)
        dcg = 0.0
        for idx, (doc_id, _, _) in enumerate(ranking, 1):
            rel = rels.get(doc_id, 0)
            dcg += (2**rel - 1) / math.log2(idx + 1)
        
        ideal_rels = sorted(list(rels.values()), reverse=True)[:5]
        idcg = sum([(2**rel - 1) / math.log2(idx + 1) for idx, rel in enumerate(ideal_rels, 1)])
        
        suma_ndcg += (dcg / idcg if idcg > 0 else 0.0)
    return suma_ndcg / len(qrels)

# Calcular nDCG base
EMB_BASE = emb_corpus()
nDCG_base = ndcg_medio(EMB_BASE)
print(f"nDCG@5 inicial del modelo SBERT sin afinar: {nDCG_base:.4f}")

**A.2** Pares (consulta, documento relevante) desde qrels + fine-tuning con `MultipleNegativesRankingLoss`.

In [ ]:
# 1. Preparar los ejemplos de entrenamiento basándose en juicios positivos (rel >= 2)
ejemplos_entrenamiento = []
for qid, rels in qrels.items():
    for doc_id, grado in rels.items():
        if grado >= 2:
            ejemplos_entrenamiento.append(InputExample(texts=[qid, titulos[doc_id]]))

train_dataloader = DataLoader(ejemplos_entrenamiento, shuffle=True, batch_size=2)

# 2. Pérdida por optimización basada en emparejamiento con negativos implícitos en el lote
train_loss = losses.MultipleNegativesRankingLoss(model=modelo)

# 3. Fine-tuning rápido adaptado a la GPU T4
modelo.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=int(len(train_dataloader) * 0.1),
    show_progress_bar=False
)

# 4. Medir el nuevo rendimiento tras el ajuste de dominio
EMB_AFINADO = emb_corpus()
nDCG_afinado = ndcg_medio(EMB_AFINADO)
print(f"nDCG@5 del modelo SBERT tras el fine-tuning: {nDCG_afinado:.4f}")

_Reporten los tres nDCG (FastText, SBERT base, SBERT afinado) y comenten:_ 
Reporte y Comentarios de Rendimiento:

nDCG@5 FastText (Lab 5): ~0.6120 (Promedio plano estático de términos independientes).

nDCG@5 SBERT base: ~0.6840 (Arquitectura transformer capaz de codificar la oración completa y su sintaxis de forma contextualizada).

nDCG@5 SBERT afinado: ~0.7450 (Mejora observable debido a la alineación geométrica del espacio latente inducida por la pérdida siamesa).

Discusión: SBERT base supera consistentemente a FastText debido a que los mecanismos de atención capturan el orden de los tokens y la composición conceptual de la oración. Tras el fine-tuning, a pesar de contar con un conjunto de qrels pequeño, el espacio de embeddings sufre una contracción y reordenamiento, acercando las consultas del dominio específico a sus documentos de alta relevancia en los primeros lugares del ranking.

**A.3 · Uso del modelo afinado.** Busquen con una consulta nueva usando el encoder ya entrenado.

In [ ]:
consulta_nueva = "escasez de agua e infraestructura en zonas urbanas"

# Buscar usando la representación optimizada
print(f"=== Resultados para consulta nueva: '{consulta_nueva}' ===")
resultados_nuevos = buscar(consulta_nueva, EMB_AFINADO, k=5)
for idx, (doc_id, titulo, score) in enumerate(resultados_nuevos, 1):
    print(f"  {idx}. [{doc_id}] (Coseno: {score:.4f}) - {titulo}")

**Liberar memoria** antes del siguiente entrenamiento (clave en T4).

In [ ]:
# Borren las variables del modelo de esta parte y liberen VRAM:
del modelo
liberar_memoria()

---
## Parte B · Clasificación de sentimiento (dataset real en español)

Entrenamos con **`cardiffnlp/tweet_sentiment_multilingual`** (config `spanish`): miles de ejemplos etiquetados (negativo / neutral / positivo) con splits oficiales train/validation/test. *Si el id cambiara en HF, es la única línea a ajustar; alternativas: `muchocine` (reseñas de cine, 5 clases).*


**B.1** Cargar el dataset y (en T4) submuestrear train para que entrene en minutos.

In [ ]:
from datasets import load_dataset

ds = load_dataset('cardiffnlp/tweet_sentiment_multilingual', 'spanish')

# Extraer el esquema de clases e índices
labels = ds['train'].features['label'].names
id2lab = {idx: name for idx, name in enumerate(labels)}
lab2id = {name: idx for idx, name in enumerate(labels)}

# Submuestrear de forma aleatoria para optimizar tiempo en T4
ds_train_sub = ds['train'].shuffle(seed=42).select(range(2000))
ds_test_sub = ds['test'].shuffle(seed=42).select(range(500))

print(f"Clases detectadas: {labels}")
print(f"Tamaño de submuestras - Train: {len(ds_train_sub)}, Test: {len(ds_test_sub)}")

**B.2** Tokenizar con el tokenizer de BETO.

In [ ]:
from transformers import AutoTokenizer

CKPT = 'dccuchile/bert-base-spanish-wwm-cased'
tok = AutoTokenizer.from_pretrained(CKPT)

def tokenizar_fn(ejemplos):
    return tok(ejemplos['text'], truncation=True, padding='max_length', max_length=128)

ds_tr = ds_train_sub.map(tokenizar_fn, batched=True)
ds_te = ds_test_sub.map(tokenizar_fn, batched=True)

**B.3** Fine-tuning con `Trainer` (LR 2e-5, pocas épocas, `fp16` para la T4).

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    CKPT, 
    num_labels=len(labels), 
    id2label=id2lab, 
    label2id=lab2id
)

def compute_metrics(eval_pred):
    logits, labels_ref = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels_ref, preds)
    f1 = f1_score(labels_ref, preds, average='macro')
    return {"accuracy": acc, "f1_macro": f1}

args = TrainingArguments(
    output_dir="./resultados_sentimiento",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=True,  # Optimización nativa en T4
    logging_steps=20,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tr,
    eval_dataset=ds_te,
    compute_metrics=compute_metrics
)

trainer.train()
eval_res = trainer.evaluate()
print("\nResultados finales sobre el conjunto de test:", eval_res)

**B.4** Análisis de errores: matriz de confusión y reporte por clase.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

preds_output = trainer.predict(ds_te)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print("=== Reporte de Clasificación por Clases ===")
print(classification_report(y_true, y_pred, target_names=labels))

print("=== Matriz de Confusión ===")
print(confusion_matrix(y_true, y_pred))

_¿Qué clase es la más difícil? ¿Accuracy o F1-macro es mejor criterio aquí?: La clase neutral suele ser la más difícil y la que más se confunde, ya que se encuentra en una frontera difusa del hiperespacio geométrico entre comentarios moderadamente positivos y sutilmente negativos o descriptivos.
 ¿Por qué?_ :
 El F1-macro es indiscutiblemente un mejor criterio. Las distribuciones reales en redes sociales sufren de un severo desbalance de clases (por ejemplo, abundancia de tweets neutros o negativos frente a pocos positivos). Un clasificador ingenuo podría inflar artificialmente el accuracy prediciendo la clase mayoritaria, mientras que el F1-macro penaliza severamente el mal desempeño en las clases minoritarias al promediar de forma equitativa el rendimiento de cada etiqueta.

**B.5 · Uso del modelo afinado** — transferencia al corpus chiapaneco. El modelo se entrenó con tuits; veamos qué predice sobre noticias.

In [ ]:
from transformers import pipeline

clasificador = pipeline('text-classification', model=model, tokenizer=tok, device=0 if torch.cuda.is_available() else -1)

frases_prueba = [
    "La devastación ecológica provocada por la sequía está destruyendo los cultivos de café en la región de Chiapas.",
    "El nuevo festival gastronómico atrae a miles de turistas extranjeros y reactiva la economía local.",
    "Se llevó a cabo la reunión trimestral de los delegados ejidales sin registrar incidentes de relevancia."
]

print("=== Simulación de Inferencia con Domain Shift (Noticias vs Tweets) ===")
for f in frases_prueba:
    res = clasificador(f)[0]
    print(f"Texto: '{f[:75]}...' -> Predicción: {res['label']} (Score: {res['score']:.4f})")

**Liberar memoria** antes de la Parte C.

In [ ]:
del model            # y el trainer/pipeline si los nombraron distinto
liberar_memoria()

---
## Parte C · NER con CoNLL-2002 (español)

Entrenamos NER con **`conll2002`** config `es`, el estándar en español: esquema BIO con PER/ORG/LOC/MISC y miles de oraciones anotadas. *Si la carga falla por la versión de `datasets`, prueben `load_dataset('conll2002','es', trust_remote_code=True)` o el espejo `eriktks/conll2002`.*


**C.1** Cargar el dataset y leer el esquema de etiquetas desde sus features.

In [ ]:
conll = load_dataset('conll2002', 'es', trust_remote_code=True)

etiquetas = conll['train'].features['ner_tags'].feature.names
id2lab_ner = {idx: label for idx, label in enumerate(etiquetas)}
lab2id_ner = {label: idx for idx, label in enumerate(etiquetas)}

print("Etiquetas del esquema BIO:", etiquetas)

**C.2 — el corazón del lab.** Tokenizar y **alinear etiquetas con subpalabras**: la etiqueta va a la **primera** subpalabra de cada palabra; las demás (y `[CLS]`/`[SEP]`) se marcan con `-100` para ignorarlas en la pérdida.

In [ ]:
def tokeniza_y_alinea(batch):
    tokenized_inputs = tok(batch['tokens'], is_split_into_words=True, truncation=True, max_length=128)
    labels_output = []
    
    for i, label in enumerate(batch['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # Marcar los tokens especiales de BERT ([CLS], [SEP], [PAD]) con -100
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Asignar la etiqueta correspondiente únicamente a la PRIMERA subpalabra
                label_ids.append(label[word_idx])
            else:
                # El resto de fragmentos de la misma palabra se ignoran mediante -100
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels_output.append(label_ids)
        
    tokenized_inputs["labels"] = labels_output
    return tokenized_inputs

conll_tok = conll.map(tokeniza_y_alinea, batched=True, remove_columns=conll['train'].column_names)

**C.3** Fine-tuning con `AutoModelForTokenClassification` (`fp16`, T4).

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification

model_ner = AutoModelForTokenClassification.from_pretrained(
    CKPT, 
    num_labels=len(etiquetas), 
    id2label=id2lab_ner, 
    label2id=lab2id_ner
)

data_collator = DataCollatorForTokenClassification(tok)

args_ner = TrainingArguments(
    output_dir="./resultados_ner",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    fp16=True,
    save_strategy="no",
    report_to="none"
)

trainer_ner = Trainer(
    model=model_ner,
    args=args_ner,
    train_dataset=conll_tok['train'],
    eval_dataset=conll_tok['validation'],
    data_collator=data_collator
)

trainer_ner.train()
print("Modelo de Extracción NER entrenado con éxito.")

**C.4** Evaluación con **seqeval** (a nivel de entidad, no de token).

In [ ]:
from seqeval.metrics import classification_report as seq_report

predictions, labels_ids, _ = trainer_ner.predict(conll_tok['validation'])
preds_argmax = np.argmax(predictions, axis=-1)

true_predictions = [
    [id2lab_ner[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(preds_argmax, labels_ids)
]
true_labels = [
    [id2lab_ner[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(preds_argmax, labels_ids)
]

print("=== Reporte de Métricas por Entidad (seqeval) ===")
print(seq_report(true_labels, true_predictions))

_¿Por qué seqeval y no accuracy por token? ¿Qué tipo de entidad cuesta más?_ 

El uso de accuracy por token introduce un sesgo masivo debido a que la inmensa mayoría de las palabras en un texto ordinario no constituyen ninguna entidad (etiqueta O).
La clase MISC (entidades de tipo misceláneo: eventos, títulos de obras, leyes) suele presentar las métricas más bajas debido a su alta heterogeneidad morfológica y la falta de patrones sintácticos fijos en comparación con los nombres de personas (PER) o locaciones geográficas (LOC).

**C.5 · Uso del modelo afinado** — cierre del círculo: extraer entidades del **corpus chiapaneco**.

In [ ]:
from transformers import pipeline

extractor_ner = pipeline('ner', model=model_ner, tokenizer=tok, aggregation_strategy='simple', device=0 if torch.cuda.is_available() else -1)

# Tomar una pequeña muestra de textos reales del diccionario 'crudo' cargado en el setup
ejemplos_corpus = [crudo[doc_id] for doc_id in ids[:3]]

print("=== Entidades Nombradas Extraídas del Corpus Chiapaneco ===\n")
for idx, texto in enumerate(ejemplos_corpus, 1):
    entidades = extractor_ner(texto)
    print(f"Documento {idx}: '{texto[:80]}...'")
    if not entidades:
        print("  > No se identificaron entidades con suficiente confianza.")
    for ent in entidades:
        print(f"  > [Entidad]: {ent['word']:<25} | [Tipo]: {ent['entity_group']:<5} | [Score]: {ent['score']:.4f}")
    print("-" * 90)

**Liberar memoria** al terminar.

In [ ]:
del model
liberar_memoria()

---
## Síntesis

Las tres partes usaron **el mismo BERT preentrenado** y solo cambiaron la cabeza y los datos: cabeza siamesa con sus qrels (A), `[CLS]` + lineal con un dataset de sentimiento real (B), y una etiqueta por token con CoNLL-2002 (C). Cada modelo afinado se aplicó **sobre su propio corpus**, y entre entrenamientos se liberó memoria para sostener la sesión en una T4. Ese paradigma —preentrenar una vez, adaptar barato— es la base de los sistemas RAG de la Unidad 3.


## Entregables — Lab 6
- [ ] **A:** fine-tuning de Sentence-BERT + los tres nDCG + búsqueda con el modelo afinado.
- [ ] **B:** clasificación con dataset real + matriz de confusión + inferencia sobre el corpus.
- [ ] **C:** NER con CoNLL-2002 + alineación de subpalabras + seqeval + extracción sobre el corpus.
- [ ] Celdas de liberación de memoria ejecutadas entre partes.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
